# Steam market analysis for Ubisoft: 03 Platforms and popularity factors
Platform availability, genre x platform, and which observable factors go together with popular games.

Popularity proxies: **estimated owners** (SteamSpy range midpoint) and **number of reviews**. They measure reach, not revenue; relations below are correlations, not causal effects.

In [0]:
%run ./setup_data

# Steam analysis: shared data setup
Loads and cleans the Steam catalog and builds the dataframes used by every analysis notebook. It is executed by the other notebooks with `%run ./setup_data` and displays nothing on purpose (lighter published notebooks). The exploration of the raw data and the justification of each cleaning step are in `00_data_exploration`.

| Dataframe | Grain | Content |
|---|---|---|
| `games_df` | 1 row per game | cleaned columns (prices, dates, owners, reviews, ages, platforms) |
| `genres_df` | 1 row per (game, genre) | `genre` exploded |
| `languages_df` | 1 row per (game, language) | `language` exploded |
| `platforms_df` | 1 row per (game, platform) | Windows / Mac / Linux |
| `tags_df` | 1 row per (game, tag) | community tags and their vote count |

### Cleaning and feature engineering
- **prices** are strings in cents: converted to USD (`try_cast` returns null instead of failing on odd values, which matters on Spark 4 / serverless where ANSI mode is on).
- **release_date** comes as `yyyy/M/d` or only `yyyy/M`: parsed to a date, `release_year` and `release_month` extracted.
- **owners** is a range (`"20,000 .. 50,000"`): lower bound, upper bound and midpoint kept. It is an estimate of sales volume (SteamSpy).
- **reviews**: total, positive ratio and the Wilson lower bound of the positive ratio (ranks games by rating while penalising small review counts).
- **required_age**: first number extracted (`"MA 15+"` becomes 15).
- **estimated_revenue_usd** = owners midpoint x current price: a rough order of magnitude, not actual sales (prices change, discounts, regional pricing).
- Only `type == "game"` is kept; duplicated `appid` are removed.

## 1. Are most games available on Windows, Mac or Linux?
*Chart: bar, X = platform, Y = pct_of_games.*

In [0]:
n_games = games_df.count()
platform_share_df = (platforms_df.groupBy("platform").agg(F.countDistinct("appid").alias("games"))
    .withColumn("pct_of_games", F.round(F.col("games") / n_games * 100, 1)).orderBy(F.desc("games")))
display(platform_share_df)

platform,games,pct_of_games
Windows,55675,100.0
Mac,12769,22.9
Linux,8457,15.2


Databricks visualization. Run in Databricks to view.

*Chart: bar, X = platforms_combo, Y = games.*

In [0]:
combo = F.concat_ws(" + ", F.when(F.col("platforms.windows"), F.lit("Windows")), F.when(F.col("platforms.mac"), F.lit("Mac")),
                    F.when(F.col("platforms.linux"), F.lit("Linux")))
combo_df = (games_df.withColumn("platforms_combo", combo).groupBy("platforms_combo").agg(F.count("*").alias("games"))
    .withColumn("pct_of_games", F.round(F.col("games") / n_games * 100, 1)).orderBy(F.desc("games")))
display(combo_df)

platforms_combo,games,pct_of_games
Windows,41271,74.1
Windows + Mac + Linux,6806,12.2
Windows + Mac,5951,10.7
Windows + Linux,1647,3.0
Mac,11,0.0
Linux,3,0.0
Mac + Linux,1,0.0


Databricks visualization. Run in Databricks to view.

## 2. Are some genres preferentially available on some platforms?
Share of the genre's games available on Mac and on Linux (genres with at least 200 games). *Chart: bar, X = genre, Y = mac_pct and linux_pct.*

In [0]:
genre_platform_df = (genres_df.groupBy("genre")
    .agg(F.countDistinct("appid").alias("games"),
         F.round(F.avg(F.col("platforms.windows").cast("int")) * 100, 1).alias("windows_pct"),
         F.round(F.avg(F.col("platforms.mac").cast("int")) * 100, 1).alias("mac_pct"),
         F.round(F.avg(F.col("platforms.linux").cast("int")) * 100, 1).alias("linux_pct"))
    .filter(F.col("games") >= 200).orderBy(F.desc("mac_pct")))
display(genre_platform_df)

genre,games,windows_pct,mac_pct,linux_pct
Strategy,10895,100.0,27.6,16.8
Indie,39681,100.0,25.0,17.6
Free to Play,3393,99.9,24.9,14.0
Design & Illustration,406,99.8,24.6,13.3
RPG,9534,100.0,23.6,16.0
Adventure,21431,100.0,23.5,15.4
Casual,22086,100.0,23.2,15.0
Animation & Modeling,322,100.0,23.0,11.8
Simulation,10836,100.0,22.5,14.1
Racing,2155,100.0,19.7,14.1


Databricks visualization. Run in Databricks to view.

## 3. What factors go together with popular games?
For each factor, the **median** estimated owners and number of reviews (medians resist the few blockbusters), and the share of games with more than 100,000 owners (lower bound of the range).

In [0]:
def popularity_by(df, col):
    return (df.groupBy(col).agg(
                F.count("*").alias("games"),
                F.percentile_approx("owners_mid", 0.5).alias("median_owners"),
                F.percentile_approx("total_reviews", 0.5).alias("median_reviews"),
                F.round(F.avg((F.col("owners_min") >= 100000).cast("int")) * 100, 2).alias("pct_100k_owners_plus"))
            .orderBy(col))

price_band = (F.when(F.col("price_usd") == 0, "0. Free").when(F.col("price_usd") < 5, "1. < $5").when(F.col("price_usd") < 10, "2. $5-10")
              .when(F.col("price_usd") < 20, "3. $10-20").when(F.col("price_usd") < 40, "4. $20-40").otherwise("5. >= $40"))
pop_df = (games_df
    .withColumn("price_band", F.when(F.col("price_usd").isNotNull(), price_band))
    .withColumn("n_languages_bucket", F.when(F.col("n_languages") <= 1, "a. 0-1").when(F.col("n_languages") <= 3, "b. 2-3")
                .when(F.col("n_languages") <= 6, "c. 4-6").when(F.col("n_languages") <= 10, "d. 7-10").otherwise("e. 11+"))
    .withColumn("n_platforms_label", F.concat(F.col("n_platforms").cast("string"), F.lit(" platform(s)")))
    .withColumn("age_restriction", F.when(F.col("required_age_int") == 0, "a. None").when(F.col("required_age_int") < 16, "b. 1-15")
                .when(F.col("required_age_int") < 18, "c. 16-17").otherwise("d. 18+")))

### Price
*Chart: bar, X = price_band, Y = pct_100k_owners_plus.*

In [0]:
display(popularity_by(pop_df.filter(F.col("price_band").isNotNull()), "price_band"))

price_band,games,median_owners,median_reviews,pct_100k_owners_plus
0. Free,7779,10000.0,47,20.0
1. < $5,23478,10000.0,15,5.89
2. $5-10,12450,10000.0,23,9.01
3. $10-20,9022,10000.0,79,18.07
4. $20-40,2394,35000.0,378,31.91
5. >= $40,567,10000.0,344,32.45


Databricks visualization. Run in Databricks to view.

### Number of languages
*Chart: bar, X = n_languages_bucket, Y = pct_100k_owners_plus.*

In [0]:
display(popularity_by(pop_df, "n_languages_bucket"))

n_languages_bucket,games,median_owners,median_reviews,pct_100k_owners_plus
a. 0-1,29665,10000.0,16,7.01
b. 2-3,10966,10000.0,23,6.88
c. 4-6,5692,10000.0,83,18.92
d. 7-10,4788,35000.0,171,27.97
e. 11+,4579,35000.0,144,30.29


Databricks visualization. Run in Databricks to view.

### Number of platforms
*Chart: bar, X = n_platforms_label, Y = pct_100k_owners_plus.*

In [0]:
display(popularity_by(pop_df, "n_platforms_label"))

n_platforms_label,games,median_owners,median_reviews,pct_100k_owners_plus
1 platform(s),41285,10000.0,21,10.0
2 platform(s),7599,10000.0,37,13.98
3 platform(s),6806,10000.0,79,21.25


Databricks visualization. Run in Databricks to view.

### Age restriction
*Chart: bar, X = age_restriction, Y = pct_100k_owners_plus.*

In [0]:
display(popularity_by(pop_df, "age_restriction"))

age_restriction,games,median_owners,median_reviews,pct_100k_owners_plus
a. None,55029,10000.0,26,11.48
b. 1-15,355,350000.0,2551,61.41
c. 16-17,76,75000.0,561,42.11
d. 18+,230,35000.0,161,30.43


Databricks visualization. Run in Databricks to view.

### Genre
*Chart: bar, X = genre, Y = pct_100k_owners_plus.*

In [0]:
genre_pop_df = (genres_df.join(games_df.select("appid", "owners_min", "total_reviews").withColumnRenamed("total_reviews", "reviews"), "appid")
    .groupBy("genre").agg(F.countDistinct("appid").alias("games"),
                          F.percentile_approx("owners_mid", 0.5).alias("median_owners"),
                          F.round(F.avg((F.col("owners_min") >= 100000).cast("int")) * 100, 2).alias("pct_100k_owners_plus"))
    .filter(F.col("games") >= 200).orderBy(F.desc("pct_100k_owners_plus")))
display(genre_pop_df)

genre,games,median_owners,pct_100k_owners_plus
Massively Multiplayer,1460,35000.0,32.33
Free to Play,3393,35000.0,32.3
RPG,9534,10000.0,16.54
Strategy,10895,10000.0,15.34
Action,23759,10000.0,13.58
Simulation,10836,10000.0,12.79
Racing,2155,10000.0,11.93
Adventure,21431,10000.0,11.9
Sports,2666,10000.0,10.43
Animation & Modeling,322,10000.0,9.94


Databricks visualization. Run in Databricks to view.

### Release year
*Chart: line, X = release_year, Y = pct_100k_owners_plus.* Older games had more time to accumulate owners, and the catalogue was smaller.

In [0]:
display(popularity_by(games_df.filter(F.col("release_year").between(2005, 2022)), "release_year"))

release_year,games,median_owners,median_reviews,pct_100k_owners_plus
2005,6,75000.0,940,50.0
2006,61,75000.0,574,47.54
2007,98,350000.0,948,75.51
2008,159,150000.0,433,52.83
2009,311,75000.0,425,49.2
2010,288,75000.0,253,43.4
2011,267,150000.0,699,62.92
2012,345,350000.0,918,65.8
2013,471,150000.0,1041,66.24
2014,1557,75000.0,307,36.35


Databricks visualization. Run in Databricks to view.

### Correlations
Pearson correlation between log(owners) and each numeric factor, on games with at least one review.

In [0]:
corr_base = (games_df.filter((F.col("total_reviews") > 0) & F.col("owners_mid").isNotNull() & F.col("price_usd").isNotNull())
    .withColumn("log_owners", F.log1p("owners_mid")).withColumn("log_reviews", F.log1p("total_reviews"))
    .withColumn("game_age_years", F.lit(2022) - F.col("release_year")))
factors = ["price_usd", "discount_pct", "n_languages", "n_platforms", "n_genres", "required_age_int", "positive_ratio", "game_age_years", "log_reviews"]
std_row = corr_base.agg(*[F.stddev(f).alias(f) for f in factors]).first()
factors = [f for f in factors if std_row[f]]  # skip constant columns (correlation undefined)
corr_row = corr_base.agg(*[F.corr("log_owners", f).alias(f) for f in factors]).first()
corr_df = spark.createDataFrame(
    [(f, round(float(corr_row[f]), 3)) for f in factors if corr_row[f] is not None],
    "factor string, correlation_with_log_owners double").orderBy(F.desc("correlation_with_log_owners"))
display(corr_df)

factor,correlation_with_log_owners
log_reviews,0.779
game_age_years,0.376
n_languages,0.215
price_usd,0.139
n_platforms,0.138
required_age_int,0.109
positive_ratio,0.085
discount_pct,0.039
n_genres,-0.01


Databricks visualization. Run in Databricks to view.

## Key figures (computed from the data)

In [0]:
ps = {r["platform"]: r["pct_of_games"] for r in platform_share_df.collect()}
mac_top = genre_platform_df.first(); mac_low = genre_platform_df.orderBy("mac_pct").first()
cr = corr_df.collect()
print(f"- Platforms: Windows {ps.get('Windows')}%, Mac {ps.get('Mac')}%, Linux {ps.get('Linux')}% of games.")
print(f"- Genre most often on Mac: {mac_top['genre']} ({mac_top['mac_pct']}%); least: {mac_low['genre']} ({mac_low['mac_pct']}%).")
print("- Correlation with log(owners): " + ", ".join(f"{r['factor']} {r['correlation_with_log_owners']:+.2f}" for r in cr))

- Platforms: Windows 100.0%, Mac 22.9%, Linux 15.2% of games.
- Genre most often on Mac: Strategy (27.6%); least: Video Production (11.7%).
- Correlation with log(owners): log_reviews +0.78, game_age_years +0.38, n_languages +0.21, price_usd +0.14, n_platforms +0.14, required_age_int +0.11, positive_ratio +0.09, discount_pct +0.04, n_genres -0.01


## How to read these results
- These are associations, not causal effects: large publishers localise and port games *because* they expect a large audience, so part of any link between languages / platforms and owners runs in the reverse direction.
- Owners come from SteamSpy ranges (midpoint used): precise enough to compare groups of games, not individual games.